# Phase 2 Proof of Concept: Search Engine Data Structures

**Name:** Sai Venkata Bharath Reddy Singareddy

**Course:** Algorithms and Data Structures (MSCS-532-B01)  
**Project Theme:** Simplified Search Engine / Search Engine Optimization  


## Phase 1 to Phase 2 connection

In Phase 1, the system design selected four main structures:

- a **dictionary** to store raw documents by document ID,
- an **inverted index** to map each term to the documents that contain it,
- a **trie** to support autocomplete using prefixes, and
- a **heap-based ranking step** to return the top results efficiently.

This Phase 2 proof of concept keeps the same design and implements the most important operations from that plan.  
The goal is **not** to build a full search engine yet. The goal is to show that the core structure works correctly and can be extended in later phases.

## Notebook roadmap

### Part 1. Foundations
I import the required Python tools and define a shared tokenization rule.

### Part 2. Trie implementation
I build the autocomplete structure in small steps.

### Part 3. Inverted index implementation
I implement document insertion, deletion, and keyword search.

### Part 4. Search engine wrapper and demonstration
I combine the trie and inverted index into one clean interface and run a small demo.

### Part 5. Test cases
I validate correctness, edge cases, and error handling.

In [1]:
from __future__ import annotations

import heapq
import re
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from typing import Dict, List, Tuple

## Shared tokenization rule

A search engine depends on a consistent rule for turning raw text into searchable terms.

For this proof of concept, the tokenizer is intentionally simple:

- convert text to lowercase,
- extract word-like tokens,
- ignore punctuation.

This keeps the implementation readable while still being realistic enough for a classroom proof of concept.

### Why this matters
If tokenization is inconsistent, then insertion and search will disagree about what counts as a term.

In [2]:
def tokenize_text(text: str) -> List[str]:
    """Convert text into lowercase searchable tokens."""
    if not isinstance(text, str):
        raise TypeError("Text must be a string.")
    return re.findall(r"\b\w+\b", text.lower())

## Trie design reasoning

The trie is responsible for **prefix-based autocomplete**.

This matches the Phase 1 requirement that users should receive suggestions while typing a query prefix.  
A trie is a good fit because its traversal cost depends mostly on the **length of the prefix**, not on the total number of stored words.

For the proof of concept, the trie supports:

- insertion of words,
- exact word checking,
- autocomplete,
- deletion.

Deletion is included because it demonstrates that the structure can stay synchronized with document updates.

In [3]:
@dataclass
class TrieNode:
    """Single trie node used for prefix search."""
    children: Dict[str, "TrieNode"] = field(default_factory=dict)
    is_word: bool = False

### Step 1: basic trie navigation and insertion

This first trie cell handles the basic structure:

- constructor,
- insert,
- exact lookup,
- internal node traversal.

Keeping these methods together makes the core navigation logic easier to inspect.

In [4]:
class Trie:
    """Trie supporting insertion, membership, autocomplete, and deletion."""

    def __init__(self) -> None:
        self.root = TrieNode()

    def insert(self, word: str) -> None:
        if not isinstance(word, str) or not word.strip():
            raise ValueError("Word must be a non-empty string.")

        node = self.root
        for ch in word.lower():
            node = node.children.setdefault(ch, TrieNode())
        node.is_word = True

    def _find_node(self, prefix: str) -> TrieNode | None:
        node = self.root
        for ch in prefix.lower():
            if ch not in node.children:
                return None
            node = node.children[ch]
        return node

    def contains(self, word: str) -> bool:
        if not isinstance(word, str) or not word.strip():
            return False
        node = self._find_node(word)
        return bool(node and node.is_word)

### Step 2: autocomplete and deletion

This second trie cell adds the two more interesting operations.

**Autocomplete** collects words below a prefix node.  
**Deletion** removes a word and prunes unused branches so the trie does not keep dead paths.

This is useful for showing that the structure is not just static; it can be updated as documents are removed.

In [5]:
def _trie_collect(node: TrieNode, prefix: str, output: List[str], limit: int) -> None:
    if len(output) >= limit:
        return

    if node.is_word:
        output.append(prefix)

    for ch in sorted(node.children.keys()):
        if len(output) >= limit:
            break
        _trie_collect(node.children[ch], prefix + ch, output, limit)


def _trie_delete_recursive(node: TrieNode, word: str, depth: int) -> bool:
    if depth == len(word):
        if not node.is_word:
            return False
        node.is_word = False
        return len(node.children) == 0

    ch = word[depth]
    child = node.children.get(ch)
    if child is None:
        return False

    should_delete_child = _trie_delete_recursive(child, word, depth + 1)

    if should_delete_child:
        del node.children[ch]

    return (not node.is_word) and (len(node.children) == 0)


def trie_autocomplete(self, prefix: str, limit: int = 10) -> List[str]:
    if not isinstance(prefix, str):
        raise TypeError("Prefix must be a string.")
    if limit <= 0:
        return []

    node = self._find_node(prefix)
    if node is None:
        return []

    results: List[str] = []
    _trie_collect(node, prefix.lower(), results, limit)
    return results


def trie_delete(self, word: str) -> bool:
    if not isinstance(word, str) or not word.strip():
        return False
    word = word.lower()
    if not self.contains(word):
        return False
    _trie_delete_recursive(self.root, word, 0)
    return True


Trie.autocomplete = trie_autocomplete
Trie.delete = trie_delete

## Inverted index design reasoning

The inverted index is the main structure for keyword search.

Instead of scanning every document every time a query is entered, the index stores a mapping like:

`term -> {doc_id: frequency}`

That lets the system jump directly to the documents that contain a term.

For this proof of concept, the inverted index supports:

- adding a document,
- removing a document,
- searching by query terms,
- tracking document storage.

### Why frequency is used
This version uses a simple term-frequency score because it is easy to explain and test.  
A later phase could replace this with TF-IDF or BM25 for more realistic ranking.

### Step 1: storage, tokenization, and document insertion

This part builds the base state of the index and shows how documents are transformed into term counts.

In [6]:
class InvertedIndex:
    """Inverted index mapping terms to document IDs and term frequencies."""

    def __init__(self) -> None:
        self.index: Dict[str, Dict[int, int]] = defaultdict(dict)
        self.documents: Dict[int, str] = {}

    def tokenize(self, text: str) -> List[str]:
        return tokenize_text(text)

    def add_document(self, doc_id: int, text: str) -> None:
        if not isinstance(doc_id, int):
            raise TypeError("Document ID must be an integer.")
        if not isinstance(text, str) or not text.strip():
            raise ValueError("Document text must be a non-empty string.")
        if doc_id in self.documents:
            raise ValueError(f"Document ID {doc_id} already exists.")

        self.documents[doc_id] = text
        term_counts = Counter(self.tokenize(text))

        for term, count in term_counts.items():
            self.index[term][doc_id] = count

### Step 2: deletion, searching, and index statistics

This cell adds the dynamic behavior of the index.

**Deletion** removes document references from posting lists.  
**Search** accumulates scores by summing matching term frequencies.  
The small helper methods make the demo and testing sections easier to read.

In [7]:
def inverted_remove_document(self, doc_id: int) -> bool:
    if doc_id not in self.documents:
        return False

    text = self.documents.pop(doc_id)
    term_counts = Counter(self.tokenize(text))

    for term in term_counts:
        posting_list = self.index.get(term, {})
        posting_list.pop(doc_id, None)
        if not posting_list and term in self.index:
            del self.index[term]

    return True


def inverted_search(self, query: str) -> Dict[int, int]:
    if not isinstance(query, str):
        raise TypeError("Query must be a string.")

    scores: Dict[int, int] = defaultdict(int)

    for term in self.tokenize(query):
        for doc_id, frequency in self.index.get(term, {}).items():
            scores[doc_id] += frequency

    return dict(scores)


def inverted_document_count(self) -> int:
    return len(self.documents)


def inverted_vocabulary_size(self) -> int:
    return len(self.index)


InvertedIndex.remove_document = inverted_remove_document
InvertedIndex.search = inverted_search
InvertedIndex.document_count = inverted_document_count
InvertedIndex.vocabulary_size = inverted_vocabulary_size

## Search engine wrapper reasoning

The trie and inverted index are useful on their own, but the application needs one simple interface.

The `SearchEngine` class acts as a **facade**:

- it stores documents,
- routes token information into both structures,
- returns ranked results,
- exposes autocomplete suggestions.


In [8]:
class SearchEngine:
    """High-level wrapper around the trie and inverted index."""

    def __init__(self) -> None:
        self.inverted_index = InvertedIndex()
        self.trie = Trie()

    def add_document(self, doc_id: int, text: str) -> None:
        self.inverted_index.add_document(doc_id, text)
        for term in set(self.inverted_index.tokenize(text)):
            self.trie.insert(term)

    def remove_document(self, doc_id: int) -> bool:
        if doc_id not in self.inverted_index.documents:
            return False

        removed_text = self.inverted_index.documents[doc_id]
        removed_terms = set(self.inverted_index.tokenize(removed_text))

        removed = self.inverted_index.remove_document(doc_id)

        if removed:
            for term in removed_terms:
                if term not in self.inverted_index.index:
                    self.trie.delete(term)

        return removed

    def search(self, query: str, k: int = 5) -> List[Tuple[int, int, str]]:
        if k <= 0:
            return []

        scores = self.inverted_index.search(query)
        top_hits = heapq.nlargest(k, scores.items(), key=lambda item: item[1])

        return [
            (doc_id, score, self.inverted_index.documents[doc_id])
            for doc_id, score in top_hits
        ]

    def suggest(self, prefix: str, limit: int = 10) -> List[str]:
        return self.trie.autocomplete(prefix, limit=limit)

    def document_count(self) -> int:
        return self.inverted_index.document_count()

    def vocabulary_size(self) -> int:
        return self.inverted_index.vocabulary_size()

## Demo dataset

This sample documents are intentionally chosen so that we can demonstrate:

- repeated terms,
- overlapping vocabulary,
- autocomplete behavior,
- changes after deletion.

In [9]:
demo_documents = {
    1: "Python makes search engine prototypes easy to build.",
    2: "Search engines use inverted indexes for fast keyword search.",
    3: "Autocomplete in Python can be implemented using a trie.",
    4: "Ranking search results with heaps is efficient for top k retrieval.",
    5: "Trie and inverted index structures support efficient search operations."
}

### Build the engine

This cell creates the search engine and loads the demo documents.

In [10]:
engine = SearchEngine()

for doc_id, text in demo_documents.items():
    engine.add_document(doc_id, text)

print("Engine built successfully.")
print("Indexed documents:", engine.document_count())
print("Vocabulary size:", engine.vocabulary_size())

Engine built successfully.
Indexed documents: 5
Vocabulary size: 37


## Demonstration 1: keyword search

This is the main retrieval behavior of the application.

The query below is designed to match multiple documents so we can see simple relevance scoring in action.

In [11]:
query = "python search"
results = engine.search(query, k=3)

print(f"Query: {query!r}")
print("Top results:")
for doc_id, score, text in results:
    print(f"- doc_id={doc_id}, score={score}, text={text}")

Query: 'python search'
Top results:
- doc_id=1, score=2, text=Python makes search engine prototypes easy to build.
- doc_id=2, score=2, text=Search engines use inverted indexes for fast keyword search.
- doc_id=3, score=1, text=Autocomplete in Python can be implemented using a trie.


### Discussion

The results show the proof-of-concept ranking logic.  
Documents that match more query terms, or match them more often, receive higher scores.

This is intentionally simpler than a production search engine, but it demonstrates the core value of the inverted index:
the system only examines documents connected to the query terms rather than scanning all documents.

## Demonstration 2: autocomplete

Now we test prefix-based suggestions.

In [12]:
prefix = "py"
suggestions = engine.suggest(prefix, limit=10)

print(f"Prefix: {prefix!r}")
print("Suggestions:", suggestions)

Prefix: 'py'
Suggestions: ['python']


### Discussion

This demonstrates the role of the trie.  
The system follows the prefix path character by character and then collects completions below that node.

That behavior directly supports the Phase 1 autocomplete goal.

## Demonstration 3: deletion and update behavior

A proof of concept is stronger when it shows that the data structure can change after initial construction.

The next cell removes one document and then shows how the search results change.

In [13]:
removed = engine.remove_document(3)

print("Document 3 removed:", removed)
print("Indexed documents after deletion:", engine.document_count())

updated_results = engine.search("python trie", k=5)
print("\nUpdated results for query 'python trie':")
for doc_id, score, text in updated_results:
    print(f"- doc_id={doc_id}, score={score}, text={text}")

Document 3 removed: True
Indexed documents after deletion: 4

Updated results for query 'python trie':
- doc_id=1, score=1, text=Python makes search engine prototypes easy to build.
- doc_id=5, score=1, text=Trie and inverted index structures support efficient search operations.


### Discussion

This step is important because it shows that the application is not just a static index.

When a document is removed:

- the document store is updated,
- the inverted index posting lists are updated,
- and trie entries that are no longer needed can be removed.

That makes the structure more realistic and easier to extend later.

## Testing strategy

The assignment asks for test cases, edge cases, and robustness.

Instead of putting all tests into one large block, the tests below are divided into smaller groups:

1. trie tests,
2. inverted index tests,
3. search engine integration tests,
4. error-handling and edge-case tests.

This makes the notebook easier to review.

### Test group 1: trie behavior

In [14]:
trie = Trie()
trie.insert("python")
trie.insert("pycharm")
trie.insert("pyramid")

assert trie.contains("python") is True
assert trie.contains("java") is False
assert trie.autocomplete("py", limit=5) == ["pycharm", "pyramid", "python"]
assert trie.delete("python") is True
assert trie.contains("python") is False

print("Trie tests passed.")

Trie tests passed.


### Why these trie tests matter

These checks verify the four most important trie behaviors:

- insertion,
- exact membership,
- autocomplete,
- deletion.

Together, they show that the autocomplete structure is functioning correctly.

### Test group 2: inverted index behavior

In [15]:
idx = InvertedIndex()
idx.add_document(1, "Python search search")
idx.add_document(2, "Trie search engine")

scores = idx.search("search")
assert scores[1] == 2
assert scores[2] == 1
assert idx.document_count() == 2
assert idx.vocabulary_size() >= 4

removed = idx.remove_document(1)
assert removed is True
assert 1 not in idx.documents
assert 1 not in idx.search("search")

print("Inverted index tests passed.")

Inverted index tests passed.


### Why these inverted index tests matter

These tests show that:

- term frequencies are stored correctly,
- document deletion updates posting lists,
- the search function returns only valid remaining documents.

This is the core behavior behind efficient keyword retrieval.

### Test group 3: search engine integration

In [16]:
test_engine = SearchEngine()
test_engine.add_document(1, "Python trie search")
test_engine.add_document(2, "Search engine ranking")
test_engine.add_document(3, "Trie autocomplete in python")

hits = test_engine.search("python trie", k=5)
hit_ids = [doc_id for doc_id, _, _ in hits]

assert 1 in hit_ids
assert 3 in hit_ids
assert "python" in test_engine.suggest("py", limit=10)

test_engine.remove_document(3)
assert 3 not in [doc_id for doc_id, _, _ in test_engine.search("python trie", k=5)]

print("Search engine integration tests passed.")

Search engine integration tests passed.


### Why the integration tests matter

The earlier tests validate components separately.  
These checks validate that the components also work **together**, which is what matters at the application level.

### Test group 4: edge cases and error handling

In [17]:
# Empty or invalid search behaviors
assert test_engine.search("anything", k=0) == []
assert test_engine.suggest("zz", limit=10) == []
assert test_engine.remove_document(999) is False

# Invalid document text
try:
    test_engine.add_document(10, "")
    raise AssertionError("Expected ValueError for empty document text.")
except ValueError:
    pass

# Invalid tokenization input
try:
    tokenize_text(123)
    raise AssertionError("Expected TypeError for non-string tokenization input.")
except TypeError:
    pass

print("Edge-case tests passed.")

Edge-case tests passed.


## Implementation challenges and reasoning

A few design choices were intentionally kept simple in this proof of concept.

### 1. Ranking model
The current ranking method is based on raw term frequency.  
This is easy to implement and easy to explain, but it is not as strong as TF-IDF or BM25.

### 2. Tokenization
The tokenizer lowercases text and removes punctuation, but it does not yet handle:
- stop-word removal,
- stemming,
- lemmatization,
- spelling correction.

### 3. Trie maintenance after deletion
If a term no longer appears in any document, the term should be removable from the trie.  
That behavior is implemented here, but it would need more extensive testing in a larger system.

### 4. Scope of the proof of concept
This notebook focuses on correctness and modular design, not scale optimization.
That is appropriate for Phase 2 because the assignment emphasizes core functionality rather than a fully finished application.

## Next steps for the full application

To move from proof of concept to a more complete system, the next development steps would be:

1. improve ranking using TF-IDF or BM25,
2. add stop-word filtering and stemming,
3. load documents from files instead of hard-coded examples,
4. build a small user interface or command-line interface,
5. add persistent storage so the index can be reused,
6. expand testing with larger datasets and performance measurements.

These next steps follow naturally from the structure built in this notebook.

## Conclusion

This notebook demonstrates the core functionality requested for Deliverable 2:

- partial implementation of the designed data structures,
- demonstration of key operations,
- test cases and edge-case handling,
- discussion of design reasoning and future work.

The implementation remains modular, readable, and extendable, which supports later phases of development.